# Master Portfolio Backtest — $100M NAV Reference

Combines the four completed strategy modules into a single portfolio, applies vol targeting,
and reports Sharpe, Max DD, leverage, and attribution across full / IS / OOS periods.

**Prerequisites:** Run each strategy notebook first so their output CSVs exist:
1. `01_momentum_research.ipynb` (or LOCAL variant) → `outputs/tsmom_portfolio_equity_*.csv`
2. `silver_gold_lease_curve_v2.ipynb` → `outputs/06_portfolio_pnl.csv`
3. `comex_efp_expansion.ipynb` → `outputs_comex/06_combined_portfolio.csv`
4. `comex_cash_and_carry.ipynb` → `outputs_comex/cc_trades_gc.csv` + `cc_trades_si.csv`

In [ ]:
## Imports & Config
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import glob
import warnings

# ── CAPITAL ALLOCATION ─────────────────────────────────────────────────────────
NAV_USD         = 100_000_000           # $100M reference book
CAPITAL_WEIGHTS = {'tsmom': 0.60, 'carry': 0.10, 'lease': 0.20, 'efp': 0.10}
RISK_BUDGETS    = {'tsmom': 0.30, 'carry': 0.10, 'lease': 0.20, 'efp': 0.20}
STRATEGY_LABELS = {
    'tsmom': 'Time-Series Momentum',
    'carry': 'Cash & Carry (COMEX)',
    'lease': 'Lease Rate Curve',
    'efp':   'COMEX EFP Basis',
}

# ── VOL TARGETING ──────────────────────────────────────────────────────────────
TARGET_STRAT_VOL = 0.10    # per-strategy annualised vol target before weighting
TARGET_PORT_VOL  = 0.10    # combined portfolio vol target
EWMA_LAMBDA      = 0.94    # EWMA decay (consistent with strategy notebooks)
LEV_CAP          = 2.0     # per-strategy max scale multiplier
PORT_LEV_CAP     = 2.0     # portfolio-level max scale multiplier
ANN              = 252     # trading days per year

# ── BACKTEST PERIODS ───────────────────────────────────────────────────────────
FULL_START = pd.Timestamp('2018-01-01')
IS_END     = pd.Timestamp('2022-12-31')
OOS_START  = pd.Timestamp('2023-01-01')
STRESS_PERIODS = {
    'COVID (2020-03)':       ('2020-03-01', '2020-06-30'),
    'Russia/Ukraine (2022)': ('2022-02-24', '2022-06-30'),
    'SVB (2023-03)':         ('2023-03-01', '2023-05-31'),
    'Tariff Shock (2025)':   ('2025-03-01', '2025-04-30'),
}
PAUSE_DD_THRESHOLD = 0.10   # flag / review if portfolio drawdown exceeds this
HARD_CAP_DD        = 0.18   # hard stop threshold (system pause)

# ── PATHS ──────────────────────────────────────────────────────────────────────
OUT_DIR   = Path('outputs')
COMEX_DIR = Path('outputs_comex')
MB_DIR    = Path('outputs_masterbacktest')
MB_DIR.mkdir(exist_ok=True)

print(f'NAV: ${NAV_USD/1e6:.0f}M')
print(f'Capital weights: {CAPITAL_WEIGHTS}')
print(f'Output directory: {MB_DIR.resolve()}')

In [ ]:
## Load: TSMOM
# Globs for latest datestamped file. Prefers BQL version (no '_local_' suffix) if present.
# Falls back to LOCAL version for offline/CSV-based testing.
all_files    = sorted(glob.glob(str(OUT_DIR / 'tsmom_portfolio_equity_*.csv')))
bql_files    = [f for f in all_files if '_local_' not in f]
local_files  = [f for f in all_files if '_local_' in f]
tsmom_candidates = bql_files or local_files  # prefer BQL

if not tsmom_candidates:
    warnings.warn('TSMOM equity file not found — sleeve zeroed out')
    r_tsmom = None
else:
    tsmom_path = tsmom_candidates[-1]  # latest by lexicographic sort
    tag = '(LOCAL)' if '_local_' in tsmom_path else '(BQL)'
    print(f'Loading TSMOM {tag}: {Path(tsmom_path).name}')
    df_ts   = pd.read_csv(tsmom_path, parse_dates=['date'], index_col='date')
    r_tsmom = df_ts['portfolio_ret_net'].rename('tsmom')
    print(f'  {r_tsmom.index[0].date()} → {r_tsmom.index[-1].date()}  '
          f'({(r_tsmom != 0).sum()} non-zero days)')

In [ ]:
## Load: Lease Rate Curve
# Assumption: pnl_bp represents daily return in basis points on the notional exposure.
# Dividing by 10_000 converts to decimal return on allocated capital (strategy sized at ~1x).
lease_path = OUT_DIR / '06_portfolio_pnl.csv'
if not lease_path.exists():
    warnings.warn(f'Lease portfolio not found at {lease_path} — sleeve zeroed out')
    r_lease = None
else:
    df_l    = pd.read_csv(lease_path, parse_dates=['date'], index_col='date')
    r_lease = (df_l['pnl_bp'] / 10_000).rename('lease')
    print(f'Lease: {r_lease.index[0].date()} → {r_lease.index[-1].date()}  '
          f'({(r_lease != 0).sum()} non-zero days)')

In [ ]:
## Load: COMEX EFP
efp_path = COMEX_DIR / '06_combined_portfolio.csv'
if not efp_path.exists():
    warnings.warn(f'EFP portfolio not found at {efp_path} — sleeve zeroed out')
    r_efp = None
else:
    df_e  = pd.read_csv(efp_path, parse_dates=['date'], index_col='date')
    r_efp = (df_e['pnl_bp'] / 10_000).rename('efp')
    print(f'EFP: {r_efp.index[0].date()} → {r_efp.index[-1].date()}  '
          f'({(r_efp != 0).sum()} non-zero days)')

In [ ]:
## Load: Cash & Carry (reconstruct daily returns from trade log)
# The C&C notebook does not save a daily return series — only individual trade legs.
# Reconstruction: distribute totalpnlbp uniformly across each trade's hold period.
# This is a smoothing simplification adequate for portfolio-level metrics.

def reconstruct_cc_daily(
    trades_path: Path,
    date_index: pd.DatetimeIndex,
) -> pd.Series:
    """Convert C&C trade log to daily decimal return series.

    Args:
        trades_path: Path to cc_trades_{metal}.csv
        date_index:  Business-day date index to align to

    Returns:
        Daily decimal return series (pnl_bp distributed uniformly over holddays).
    """
    df  = pd.read_csv(trades_path, parse_dates=['entrydate', 'exitdate'])
    pnl = pd.Series(0.0, index=date_index, dtype=float)
    for _, row in df.iterrows():
        mask = (date_index >= row['entrydate']) & (date_index <= row['exitdate'])
        n = int(mask.sum())
        if n > 0:
            pnl[mask] += row['totalpnlbp'] / n
    return pnl / 10_000  # bp → decimal return

date_index = pd.bdate_range(start=FULL_START, end=pd.Timestamp.today())

gc_path = COMEX_DIR / 'cc_trades_gc.csv'
si_path = COMEX_DIR / 'cc_trades_si.csv'

if gc_path.exists() and si_path.exists():
    r_gc    = reconstruct_cc_daily(gc_path, date_index)
    r_si    = reconstruct_cc_daily(si_path, date_index)
    # Equal-weight GC and SI within the C&C sleeve
    r_carry = ((r_gc + r_si) / 2).rename('carry')
    print(f'C&C GC: {(r_gc != 0).sum()} non-zero days')
    print(f'C&C SI: {(r_si != 0).sum()} non-zero days')
    print(f'C&C combined: {(r_carry != 0).sum()} non-zero days')
else:
    warnings.warn(f'C&C trade logs not found in {COMEX_DIR} — sleeve zeroed out')
    r_carry = None

In [ ]:
## Align: build common daily return panel
raw_series = {
    'tsmom': r_tsmom,
    'carry': r_carry,
    'lease': r_lease,
    'efp':   r_efp,
}
available = {k: v for k, v in raw_series.items() if v is not None}

# Align to business-day index from FULL_START; fill NaN with 0 (no-trade = zero return)
rets = pd.DataFrame(available).reindex(date_index).fillna(0.0)
rets = rets.loc[FULL_START:]
rets.index.name = 'date'

# Zero-out any stray NaN remaining at boundaries
rets = rets.fillna(0.0)

print('\nReturn panel coverage:')
print(f'{"Strategy":<12}  {"First":>12}  {"Last":>12}  {"Non-zero days":>15}')
print('-' * 55)
for col in rets.columns:
    nz  = (rets[col] != 0).sum()
    nz_slice = rets[col][rets[col] != 0]
    first_nz = nz_slice.index[0].date() if len(nz_slice) else 'n/a'
    last_nz  = nz_slice.index[-1].date() if len(nz_slice) else 'n/a'
    print(f'{col:<12}  {str(first_nz):>12}  {str(last_nz):>12}  {nz:>15,}')

missing_strats = [k for k, v in raw_series.items() if v is None]
if missing_strats:
    print(f'\n  ⚠ Missing strategies (zeroed): {missing_strats}')

In [ ]:
## Vol Scaling: per-strategy EWMA targeting
# Each strategy is independently scaled to TARGET_STRAT_VOL before capital weighting.
# Scale uses one-period-lagged vol to avoid lookahead bias.

def ewma_vol(
    r: pd.Series,
    lam: float = EWMA_LAMBDA,
    ann: int   = ANN,
    min_periods: int = 20,
) -> pd.Series:
    """Annualised EWMA realised volatility, lagged one period.

    Args:
        r:           Daily return series.
        lam:         EWMA decay parameter (default 0.94).
        ann:         Annualisation factor (252 trading days).
        min_periods: Minimum observations before returning non-NaN.

    Returns:
        Annualised vol series (shifted one day to avoid lookahead).
    """
    com   = (1 - lam) / lam
    var   = r.ewm(com=com, min_periods=min_periods).var()
    vol   = (var * ann).pow(0.5)
    return vol.shift(1).fillna(TARGET_STRAT_VOL)

scaled_dict  = {}
scale_dict   = {}  # keep for gross leverage computation
vol_dict     = {}

for strat, r in rets.items():
    vol   = ewma_vol(r)
    scale = (TARGET_STRAT_VOL / vol).clip(upper=LEV_CAP).fillna(1.0)
    scale_dict[strat]  = scale
    vol_dict[strat]    = vol
    scaled_dict[strat] = r * scale

rets_scaled = pd.DataFrame(scaled_dict)

# Sanity check: vol of scaled series should be near TARGET_STRAT_VOL
print('Post-scaling annualised vol check (should be ≈ 10%):')     
for col in rets_scaled.columns:
    rv = rets_scaled[col].std() * ANN**0.5
    print(f'  {col:<10s}: {rv:.1%}')

In [ ]:
## Portfolio: combine strategies + NAV equity curve

# 1. Capital-weighted sum of vol-scaled strategy returns
port_ret = pd.Series(0.0, index=rets_scaled.index)
for strat, r in rets_scaled.items():
    port_ret += CAPITAL_WEIGHTS.get(strat, 0.0) * r

# 2. Portfolio-level vol overlay to hit TARGET_PORT_VOL
port_vol   = ewma_vol(port_ret)
port_scale = (TARGET_PORT_VOL / port_vol).clip(upper=PORT_LEV_CAP).fillna(1.0)
port_ret_f = (port_ret * port_scale).rename('portfolio')

# 3. NAV equity curve — compound daily returns on $100M starting capital
nav        = NAV_USD * (1 + port_ret_f).cumprod()
nav_df     = pd.DataFrame({
    'nav_usd':  nav,
    'ret_net':  port_ret_f,
}, index=port_ret_f.index)

# 4. Gross leverage time series
# = Σ(capital_weight_i × per_strat_scale_i) × portfolio_scale
gross_lev = sum(
    CAPITAL_WEIGHTS.get(s, 0.0) * scale_dict[s]
    for s in scale_dict
) * port_scale

print(f'Combined portfolio vol (realised): {port_ret_f.std() * ANN**0.5:.1%}')
print(f'Avg gross leverage:                {gross_lev.mean():.2f}x')
print(f'Max gross leverage:                {gross_lev.max():.2f}x')
print(f'Final NAV (end of period):         ${nav.iloc[-1]/1e6:.1f}M')

In [ ]:
## Metrics: full / IS / OOS + per-strategy + stress

def compute_metrics(r: pd.Series) -> dict:
    """Standard portfolio performance metrics for a daily return series."""
    r = r.dropna()
    if len(r) < 20:
        return {}
    ann_ret  = float(r.mean() * ANN)
    ann_vol  = float(r.std() * ANN**0.5)
    sharpe   = ann_ret / ann_vol if ann_vol > 0 else 0.0
    cum      = (1 + r).cumprod()
    roll_max = cum.cummax()
    dd       = cum / roll_max - 1
    max_dd   = float(dd.min())
    calmar   = ann_ret / abs(max_dd) if max_dd != 0 else 0.0
    hit_rate = float((r > 0).mean())
    pos_ret  = r[r > 0].mean() if (r > 0).any() else np.nan
    neg_ret  = r[r < 0].mean() if (r < 0).any() else np.nan
    win_loss = float(pos_ret / abs(neg_ret)) if not np.isnan(pos_ret) and neg_ret != 0 else np.nan
    return dict(
        ann_return=ann_ret,
        ann_vol=ann_vol,
        sharpe=sharpe,
        max_dd=max_dd,
        calmar=calmar,
        hit_rate=hit_rate,
        win_loss=win_loss,
    )

# Portfolio slices
slices = {
    'Full': port_ret_f,
    'IS':   port_ret_f.loc[:IS_END],
    'OOS':  port_ret_f.loc[OOS_START:],
}
perf    = {period: compute_metrics(r) for period, r in slices.items()}
perf_df = pd.DataFrame(perf).T

# Per-strategy attribution
strat_attr = {}
for s, r_sc in rets_scaled.items():
    r_contrib = r_sc * CAPITAL_WEIGHTS.get(s, 0.0) * port_scale
    m = compute_metrics(r_sc)
    strat_attr[STRATEGY_LABELS.get(s, s)] = {
        'capital_weight':       CAPITAL_WEIGHTS.get(s, 0),
        'risk_budget':          RISK_BUDGETS.get(s, 0),
        'ann_return':           m.get('ann_return', np.nan),
        'ann_vol':              m.get('ann_vol', np.nan),
        'sharpe':               m.get('sharpe', np.nan),
        'max_dd':               m.get('max_dd', np.nan),
        'port_contribution_ann': float(r_contrib.mean() * ANN),
    }
attr_df = pd.DataFrame(strat_attr).T

# Stress period analysis
stress_results = {}
for label, (start, end) in STRESS_PERIODS.items():
    slc = port_ret_f.loc[start:end]
    if len(slc) >= 5:
        stress_results[label] = compute_metrics(slc)
stress_df = pd.DataFrame(stress_results).T

In [ ]:
## Summary: headline metrics table

W = 64
print('\n' + '=' * W)
print(f'  MASTER PORTFOLIO — ${NAV_USD/1e6:.0f}M NAV REFERENCE BOOK')
print('=' * W)
header = f'  {"Metric":<26}  {"Full":>10}  {"IS (≤2022)":>10}  {"OOS (2023+)":>10}'
print(header)
print('-' * W)

fields = [
    ('Ann. Return (net)',  'ann_return', '{:+.1%}'),
    ('Ann. Volatility',   'ann_vol',    '{:.1%}'),
    ('Sharpe Ratio',      'sharpe',     '{:.2f}'),
    ('Max Drawdown',      'max_dd',     '{:.1%}'),
    ('Calmar Ratio',      'calmar',     '{:.2f}'),
    ('Hit Rate (daily)',  'hit_rate',   '{:.1%}'),
    ('Win/Loss Ratio',    'win_loss',   '{:.2f}'),
]
for label, key, fmt_str in fields:
    row = f'  {label:<26}'
    for period in ['Full', 'IS', 'OOS']:
        v = perf.get(period, {}).get(key, float('nan'))
        cell = fmt_str.format(v) if not np.isnan(v) else 'n/a'
        row += f'  {cell:>10}'
    print(row)

# Leverage (time-averaged, no IS/OOS split here)
avg_lev = float(gross_lev.mean())
max_lev = float(gross_lev.max())
print(f'  {"Avg Gross Leverage":<26}  {avg_lev:>10.2f}x')
print(f'  {"Max Gross Leverage":<26}  {max_lev:>10.2f}x')
print('=' * W)

# DD breach check
port_dd = ((1 + port_ret_f).cumprod() / (1 + port_ret_f).cumprod().cummax() - 1)
current_dd = float(port_dd.iloc[-1])
breach_flag = '⚠  PAUSE THRESHOLD BREACHED' if current_dd < -PAUSE_DD_THRESHOLD else 'OK'
print(f'\n  Current drawdown: {current_dd:.1%}  [{breach_flag}]')

print('\n  Per-strategy attribution (vol-scaled):')      
display(attr_df.rename(columns={
    'capital_weight': 'CapWt', 'risk_budget': 'RiskBudget',
    'ann_return': 'AnnRet', 'ann_vol': 'AnnVol',
    'sharpe': 'Sharpe', 'max_dd': 'MaxDD',
    'port_contribution_ann': 'PortContrib'
}).round(4))

if not stress_df.empty:
    print('\n  Stress period performance:')
    display(stress_df[['ann_return', 'sharpe', 'max_dd']].round(4))

In [ ]:
## Chart: NAV equity curve + drawdown

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['NAV Equity Curve ($M)', 'Drawdown (%)'],
    vertical_spacing=0.06,
)

nav_m = nav / 1e6  # in $M
fig.add_trace(go.Scatter(
    x=nav_m.index, y=nav_m.values,
    name='Portfolio NAV',
    line=dict(color='royalblue', width=2),
), row=1, col=1)

# Drawdown trace
fig.add_trace(go.Scatter(
    x=port_dd.index, y=port_dd.values * 100,
    name='Drawdown',
    fill='tozeroy',
    line=dict(color='crimson', width=1.5),
    fillcolor='rgba(220,20,60,0.15)',
), row=2, col=1)

# Pause / hard-cap DD lines
for lvl, label, dash in [
    (-PAUSE_DD_THRESHOLD * 100, f'Pause {-PAUSE_DD_THRESHOLD:.0%}', 'dash'),
    (-HARD_CAP_DD * 100,        f'Hard stop {-HARD_CAP_DD:.0%}',    'dot'),
]:
    fig.add_hline(y=lvl, row=2, col=1,
                  line=dict(color='crimson', dash=dash, width=1),
                  annotation_text=label,
                  annotation_position='left')

# IS / OOS boundary — use ISO strings, not pd.Timestamp (Plotly compatibility)
oos_str = OOS_START.strftime('%Y-%m-%d')
is_str  = IS_END.strftime('%Y-%m-%d')
for row_n in [1, 2]:
    fig.add_vline(x=oos_str, row=row_n, col=1,
                  line=dict(color='grey', dash='dot', width=1.5),
                  annotation_text='OOS →' if row_n == 1 else '',
                  annotation_position='top right')

# Stress period shading
colours = ['rgba(255,165,0,0.10)', 'rgba(128,0,128,0.10)',
           'rgba(0,128,0,0.10)',    'rgba(255,0,0,0.10)']
for (label, (s, e)), colour in zip(STRESS_PERIODS.items(), colours):
    for row_n in [1, 2]:
        fig.add_vrect(x0=s, x1=e, row=row_n, col=1,
                      fillcolor=colour, line_width=0,
                      annotation_text=label if row_n == 1 else '',
                      annotation_position='top left')

# Sharpe annotations on equity panel
sr_is  = perf.get('IS', {}).get('sharpe', np.nan)
sr_oos = perf.get('OOS', {}).get('sharpe', np.nan)
sr_ann_ret_is  = perf.get('IS', {}).get('ann_return', np.nan)
sr_ann_ret_oos = perf.get('OOS', {}).get('ann_return', np.nan)
if not np.isnan(sr_is):
    fig.add_annotation(x=is_str, y=float(nav_m.loc[:IS_END].iloc[-1]),
                       text=f'IS SR={sr_is:.2f}<br>Ret={sr_ann_ret_is:+.1%}',
                       showarrow=False, bgcolor='rgba(65,105,225,0.15)',
                       font=dict(size=11), row=1, col=1)
if not np.isnan(sr_oos):
    oos_last = port_ret_f.loc[OOS_START:].index[-1].strftime('%Y-%m-%d')
    fig.add_annotation(x=oos_last,
                       y=float(nav_m.iloc[-1]),
                       text=f'OOS SR={sr_oos:.2f}<br>Ret={sr_ann_ret_oos:+.1%}',
                       showarrow=False, bgcolor='rgba(34,139,34,0.15)',
                       font=dict(size=11), row=1, col=1)

fig.update_layout(
    title=f'Master Portfolio — ${NAV_USD/1e6:.0f}M NAV (Sharpe: {perf["Full"].get("sharpe", 0):.2f} | Max DD: {perf["Full"].get("max_dd", 0):.1%})',
    height=650, hovermode='x unified',
    legend=dict(orientation='h', y=-0.08),
    margin=dict(l=60, r=60, t=80, b=40),
)
fig.update_yaxes(title_text='NAV ($M)', row=1, col=1)
fig.update_yaxes(title_text='Drawdown (%)', row=2, col=1)
fig.show()

In [ ]:
## Chart: per-strategy cumulative returns (4-panel)

strats = list(rets_scaled.columns)
n_cols = 2
n_rows = (len(strats) + n_cols - 1) // n_cols
subplot_titles = [
    f'{STRATEGY_LABELS.get(s, s)}  |  '
    f'CapWt={CAPITAL_WEIGHTS.get(s,0):.0%}  '
    f'SR={strat_attr.get(STRATEGY_LABELS.get(s,s),{}).get("sharpe", float("nan")):.2f}  '
    f'MaxDD={strat_attr.get(STRATEGY_LABELS.get(s,s),{}).get("max_dd", float("nan")):.1%}'
    for s in strats
]

fig2 = make_subplots(rows=n_rows, cols=n_cols,
                     subplot_titles=subplot_titles,
                     shared_xaxes=True, vertical_spacing=0.10)

palette = ['royalblue', 'darkorange', 'green', 'crimson']
for i, strat in enumerate(strats):
    row_i = i // n_cols + 1
    col_i = i % n_cols + 1
    cum   = (1 + rets_scaled[strat]).cumprod() - 1
    fig2.add_trace(go.Scatter(
        x=cum.index, y=cum.values * 100,
        name=STRATEGY_LABELS.get(strat, strat),
        line=dict(color=palette[i % len(palette)], width=1.8),
        showlegend=False,
    ), row=row_i, col=col_i)
    # IS/OOS boundary
    fig2.add_vline(x=OOS_START.strftime('%Y-%m-%d'), row=row_i, col=col_i,
                   line=dict(color='grey', dash='dot', width=1))

fig2.update_yaxes(title_text='Cumulative return (%)')
fig2.update_layout(
    title='Per-Strategy Cumulative Returns (vol-scaled to 10% p.a.)',
    height=520, hovermode='x unified',
    margin=dict(l=60, r=40, t=100, b=40),
)
fig2.show()

In [ ]:
## Chart: rolling metrics + gross leverage

ROLL_WIN = 252  # 1-year rolling window

roll_sharpe = (port_ret_f.rolling(ROLL_WIN).mean() * ANN) / \
              (port_ret_f.rolling(ROLL_WIN).std() * ANN**0.5)
roll_vol    = port_ret_f.rolling(ROLL_WIN).std() * ANN**0.5 * 100  # in %

fig3 = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        f'{ROLL_WIN}-Day Rolling Sharpe',
        f'{ROLL_WIN}-Day Rolling Volatility (%)',
        'Gross Leverage (×)',
    ],
    vertical_spacing=0.07,
    row_heights=[0.36, 0.32, 0.32],
)

# Rolling Sharpe
fig3.add_trace(go.Scatter(
    x=roll_sharpe.index, y=roll_sharpe.values,
    name='Rolling Sharpe', line=dict(color='royalblue', width=1.8),
), row=1, col=1)
for threshold, dash, colour in [(0.0, 'dash', 'grey'), (1.0, 'dot', 'green')]:
    fig3.add_hline(y=threshold, row=1, col=1,
                   line=dict(color=colour, dash=dash, width=1),
                   annotation_text=str(threshold) if threshold > 0 else '0',
                   annotation_position='right')

# Rolling vol
fig3.add_trace(go.Scatter(
    x=roll_vol.index, y=roll_vol.values,
    name='Rolling Vol', line=dict(color='darkorange', width=1.8),
), row=2, col=1)
fig3.add_hline(y=TARGET_PORT_VOL * 100, row=2, col=1,
               line=dict(color='grey', dash='dot', width=1),
               annotation_text=f'Target {TARGET_PORT_VOL:.0%}',
               annotation_position='right')

# Gross leverage
fig3.add_trace(go.Scatter(
    x=gross_lev.index, y=gross_lev.values,
    name='Gross Leverage', line=dict(color='purple', width=1.8),
    fill='tozeroy', fillcolor='rgba(128,0,128,0.08)',
), row=3, col=1)
fig3.add_hline(y=8.0, row=3, col=1,
               line=dict(color='crimson', dash='dot', width=1),
               annotation_text='8× hard cap',
               annotation_position='right')

# IS/OOS line on all rows
for row_n in [1, 2, 3]:
    fig3.add_vline(x=OOS_START.strftime('%Y-%m-%d'), row=row_n, col=1,
                   line=dict(color='grey', dash='dot', width=1))

fig3.update_layout(
    title='Rolling Portfolio Analytics',
    height=680, hovermode='x unified',
    showlegend=False,
    margin=dict(l=60, r=80, t=80, b=40),
)
fig3.update_yaxes(title_text='Sharpe',   row=1, col=1)
fig3.update_yaxes(title_text='Vol (%)',  row=2, col=1)
fig3.update_yaxes(title_text='Leverage', row=3, col=1)
fig3.show()

In [ ]:
## Export: write all master outputs

# 1. Headline metrics (Full / IS / OOS)
perf_df.to_csv(MB_DIR / 'master_summary.csv')

# 2. Daily NAV and return
nav_df.to_csv(MB_DIR / 'master_nav.csv')

# 3. Per-strategy attribution
attr_df.to_csv(MB_DIR / 'master_attribution.csv')

# 4. Stress period results
if not stress_df.empty:
    stress_df.to_csv(MB_DIR / 'master_stress.csv')

# 5. Gross leverage time series
gross_lev.rename('gross_leverage').to_frame().to_csv(MB_DIR / 'master_leverage.csv')

# 6. Per-strategy scaled daily returns (for further analysis)
rets_scaled.to_csv(MB_DIR / 'master_strat_returns.csv')

written = list(MB_DIR.glob('*.csv'))
print(f'Written {len(written)} files to {MB_DIR.resolve()}:')
for f in sorted(written):
    print(f'  {f.name}')